In [2]:
"""
使用trl库进行SFT全参微调
"""
from datasets import load_dataset

# 1、数据加载
keyword_data = load_dataset("json",data_files={"train":r"./data/keywords_data_train.jsonl","test":r"./data/keywords_data_test.jsonl"})
"""
{
    "conversation_id": 1,
    "category": "dialogue",
    "conversation": [
        {
            "human": "抽取出文本中的关键词：\n标题：人工神经网络在猕猴桃种类识别上的应用\n文本：在猕猴桃介电特性研究的基础上,将人工神经网络技术应用于猕猴桃的种类识别.该种类识别属于模式识别,其关键在于提取样品的特征参数,在获得特征参数的基础上,选取合适的网络通过训练来进行识别.猕猴桃种类识别的研究为自动化识别果品的种类、品种和新鲜等级等提供了一种新方法,为进一步研究果品介电特性与其内在品质的关系提供了一定的理论与实践基础.",
            "assistant": "食品科学技术基础学科;猕猴桃;应用;人工神经网络;介电特性;识别"
        }
    ],
    "dataset": "psychology"
}
"""

D:\develop\PycharmProjects\sgg_projects\llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'\n{\n    "conversation_id": 1,\n    "category": "dialogue",\n    "conversation": [\n        {\n            "human": "抽取出文本中的关键词：\n标题：人工神经网络在猕猴桃种类识别上的应用\n文本：在猕猴桃介电特性研究的基础上,将人工神经网络技术应用于猕猴桃的种类识别.该种类识别属于模式识别,其关键在于提取样品的特征参数,在获得特征参数的基础上,选取合适的网络通过训练来进行识别.猕猴桃种类识别的研究为自动化识别果品的种类、品种和新鲜等级等提供了一种新方法,为进一步研究果品介电特性与其内在品质的关系提供了一定的理论与实践基础.",\n            "assistant": "食品科学技术基础学科;猕猴桃;应用;人工神经网络;介电特性;识别"\n        }\n    ],\n    "dataset": "psychology"\n}\n'

In [3]:
# 2. 数据格式转化函数
from typing import Dict,List
def data_convert(examples: Dict[str, List]):
    """
    详情见 unsloth.txt
    """
    conversation_example_list = examples["conversation"]
    message_text_list = []
    for example in conversation_example_list:
        message_list = []
        conversation = example[0]
        message_list.append({"role": "user", "content": conversation["human"]})
        message_list.append({"role": "assistant", "content": conversation["assistant"]})

        message_text_list.append(message_list)

    return {"messages": message_text_list}

# map 调用函数
mapped_keyword_data = keyword_data.map(data_convert,batched=True,remove_columns=keyword_data["train"].column_names)
mapped_keyword_data

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 49500
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 500
    })
})

In [4]:
mapped_keyword_data["train"][0] # 训练集第一条
"""
{
    "messages": [
        {
            'content': (
                '高氟铍矿石在熔炼过程中配入氢氧化铝来脱除其中的氟.'
                '结果表明,在配入5％Na2CO3、9.3％Al(OH)3、1400～1500℃熔炼20 min的情况下,'
                'BeO回收率达到96％以上,脱氟效果良好(铍玻璃F/BeO能控制在15％以内).'
                '为高氟铍矿石的工业应用探索出新的冶炼途径.\n找出上文中的关键词'
            ),
            'role': 'user'
        },
        {
            'content': '高氟铍矿;配料;熔炼;回收率;脱氟率',
            'role': 'assistant'
        }
    ]
}
"""

'\n{\n    "messages": [\n        {\n            \'content\': (\n                \'高氟铍矿石在熔炼过程中配入氢氧化铝来脱除其中的氟.\'\n                \'结果表明,在配入5％Na2CO3、9.3％Al(OH)3、1400～1500℃熔炼20 min的情况下,\'\n                \'BeO回收率达到96％以上,脱氟效果良好(铍玻璃F/BeO能控制在15％以内).\'\n                \'为高氟铍矿石的工业应用探索出新的冶炼途径.\n找出上文中的关键词\'\n            ),\n            \'role\': \'user\'\n        },\n        {\n            \'content\': \'高氟铍矿;配料;熔炼;回收率;脱氟率\',\n            \'role\': \'assistant\'\n        }\n    ]\n}\n'

In [5]:
mapped_keyword_data["train"][:2] # 训练集前两条
"""
{
    "messages": [
        # ==================== 第一条数据 (Sample 1) ====================
        [
            {
                'content': (
                    '高氟铍矿石在熔炼过程中配入氢氧化铝来脱除其中的氟.'
                    '结果表明,在配入5％Na2CO3、9.3％Al(OH)3、1400～1500℃熔炼20 min的情况下,'
                    'BeO回收率达到96％以上,脱氟效果良好(铍玻璃F/BeO能控制在15％以内).'
                    '为高氟铍矿石的工业应用探索出新的冶炼途径.\n找出上文中的关键词'
                ),
                'role': 'user'
            },
            {
                'content': '高氟铍矿;配料;熔炼;回收率;脱氟率',
                'role': 'assistant'
            }
        ],

        # ==================== 第二条数据 (Sample 2) ====================
        [
            {
                'content': (
                    '关键词抽取：\n目的:考察蒙药芯芭提取物的抗炎、镇痛作用.'
                    '方法:将96只KM小鼠或SD大鼠随机分为模型组(水),阳性对照组(阿司匹林,0.5 g/kg),'
                    '芯芭醇提物(70%乙醇提取)低、中、高剂量组(0.325、0.650、1.300 g/kg,以生药计)'
                    '和芯芭醇提后残渣水提物低、中、高剂量组(0.325、0.650、1.300 g/kg,以生药计),'
                    '每组12只;ig给药,每天1次,连续7 d.分别采用二甲苯致小鼠耳肿胀法测定小鼠耳肿胀度和'
                    '蛋清致大鼠足肿胀法测定大鼠致炎1、2、4、6 h后的足跖肿胀度,以考察芯芭提取物的抗炎作用.'
                    '取96只KM小鼠同法分组、给药,采用乙酸扭体法测定小鼠20 min内扭体次数;另取64只KM小鼠,'
                    '同法分组,每组8只,除阳性对照组小鼠ig盐酸曲马多(0.5 g/kg)外,其余各组小鼠同法给药,'
                    '采用热板法测定小鼠给药前及给药30、45、60、90 min后的痛阈值,以考察芯芭提取物的镇痛作用.'
                    '结果:与模型组比较,芯芭提取物可降低小鼠耳肿胀度和蛋清致炎6h后大鼠足跖肿胀度,'
                    '除芯芭醇提后残渣水提物高剂量组大鼠足跖肿胀度降低不显著外,其余各组差异均有统计学意义(P<0.05或P<0.01);'
                    '芯芭提取物可显著减少小鼠20 min内扭体次数,延长给药30、60、90 min后的小鼠痛阈值(P<0.05).'
                    '结论:芯芭醇提物及其醇提残渣水提物均具有一定的抗炎、镇痛作用.'
                ),
                'role': 'user'
            },
            {
                'content': '蒙药;芯芭;提取物;抗炎;镇痛;小鼠;大鼠',
                'role': 'assistant'
            }
        ]
    ]
}
"""

'\n{\n    "messages": [\n        # ==================== 第一条数据 (Sample 1) ====================\n        [\n            {\n                \'content\': (\n                    \'高氟铍矿石在熔炼过程中配入氢氧化铝来脱除其中的氟.\'\n                    \'结果表明,在配入5％Na2CO3、9.3％Al(OH)3、1400～1500℃熔炼20 min的情况下,\'\n                    \'BeO回收率达到96％以上,脱氟效果良好(铍玻璃F/BeO能控制在15％以内).\'\n                    \'为高氟铍矿石的工业应用探索出新的冶炼途径.\n找出上文中的关键词\'\n                ),\n                \'role\': \'user\'\n            },\n            {\n                \'content\': \'高氟铍矿;配料;熔炼;回收率;脱氟率\',\n                \'role\': \'assistant\'\n            }\n        ],\n\n        # ==================== 第二条数据 (Sample 2) ====================\n        [\n            {\n                \'content\': (\n                    \'关键词抽取：\n目的:考察蒙药芯芭提取物的抗炎、镇痛作用.\'\n                    \'方法:将96只KM小鼠或SD大鼠随机分为模型组(水),阳性对照组(阿司匹林,0.5 g/kg),\'\n                    \'芯芭醇提物(70%乙醇提取)低、中、高剂量组(0.325、0.650、1.300 g/kg,以生药计)\'\n                    \'和芯芭醇提后残渣水提物低、中、高剂量组(0.325、0

In [6]:
# 3. 构造SFTConfig实例
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl.trainer.sft_config import SFTConfig
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs/04_TRL_DEMO"
model = AutoModelForCausalLM.from_pretrained("model/Qwen3-0.6B/")
tokenizer = AutoTokenizer.from_pretrained("model/Qwen3-0.6B/")
training_args = SFTConfig(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    max_steps=1000,
    num_train_epochs=1,
    logging_strategy="steps",
    logging_steps=100,
    report_to="tensorboard",
    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    eval_strategy="steps",
    eval_steps=100,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    load_best_model_at_end=True,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    output_dir="finetuned/04_TRL_DEMO",
    bf16=True,
    max_length=710,
    assistant_only_loss=True,
    chat_template_path="./chat_template.jinja"
)

# 4. 构造trainer
from trl.trainer.sft_trainer import SFTTrainer
trainer = SFTTrainer(
    args=training_args,
    model=model,
    train_dataset=mapped_keyword_data["train"],
    eval_dataset=mapped_keyword_data["test"],
    processing_class=tokenizer
)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu130).
W0624 21:01:37.892000 30296 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 921.02it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [7]:
# 5. 获取 data_loader
data_loader = trainer.get_train_dataloader()
# 解码检查数据
for batch_data in data_loader:
    decoded_batch_data = tokenizer.decode(batch_data["input_ids"])
    print(decoded_batch_data[0])
    break

<|im_start|>system
## Metadata

Knowledge Cutoff Date: June 2025
Today Date: 24 June 2026
Reasoning Mode: /think

## Custom Instructions

You are a helpful AI assistant named SmolLM, trained by Hugging Face. Your role as an assistant involves thoroughly exploring questions through a systematic thinking process before providing the final precise and accurate solutions. This requires engaging in a comprehensive cycle of analysis, summarizing, exploration, reassessment, reflection, backtracking, and iteration to develop well-considered thinking process. Please structure your response into two main sections: Thought and Solution using the specified format: <think> Thought section </think> Solution section. In the Thought section, detail your reasoning process in steps. Each step should include detailed considerations such as analysing questions, summarizing relevant findings, brainstorming new ideas, verifying the accuracy of the current steps, refining any errors, and revisiting previous 

In [12]:
# 6. 训练
trainer.train()
trainer.save_model("finetuned/04_TRL_DEMO")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss


AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [9]:
# 扩展：获取数据长度的分布情况
def get_lenth_map(example):
    """
    获取数据的长度
    """
    # result: input_ids, attention_mask
    result:Dict = tokenizer.apply_chat_template(example["messages"],tokenize=True)
    return {"length":len(result["input_ids"])}

data_lenth = mapped_keyword_data.map(get_lenth_map,batched=False,remove_columns=["messages"])

In [10]:
# 得到train当中，第0条样本，经过了chat template处理之后，所得到input_ids的长度，是380
data_lenth["train"][0]["length"]
"""
[
    {"length": 380},  # 第 0 条
    {"length": 620},  # 第 1 条
    # ... 还有 49498 个这样的小字典 ...
]
"""

380

In [11]:
res_list = data_lenth["train"].to_list()    # Dataset 转 list
example_length = [example["length"] for example in res_list]
# [380, 620, 241, 158, 489, 750, ...]
import pandas as pd
length_series = pd.Series(example_length)
length_series.quantile(0.99)

np.float64(701.0)